In [1]:
import pypsa
import pandas as pd
import xlwings as xw

In [2]:
# 1) otwieramy plik Excela
wb = xw.Book(r"bess_model.xlsx")

# 2) bierzemy pierwszą zakładkę (albo podaj nazwę: wb.sheets['Dashboard'])
sht = wb.sheets['Dashboard']

# 3) wczytujemy całą tabelę od A1
#    index=True -> pierwsza kolumna ("Scenario", "BESS_MW", ...) jako index
#    header=True -> pierwszy wiersz (1,2,3,4,5) jako nazwy kolumn
df_cfg = sht.range("A1").expand().options(pd.DataFrame,
                                           index=True,
                                           header=True).value



sht = wb.sheets['wsad_python']
df_inp = sht.range("A1").expand().options(pd.DataFrame,
                                           index=True,
                                           header=True).value.reset_index()



In [3]:
scenario_id = 3   # możesz podmienić np. na 1,2,4,5...
  
col = df_cfg[str(scenario_id)] # to jest jedna kolumna z ustawieniami
# słownik BESS do podania w build_bess()

bess_config = {
    "name": "battery",
    "p_nom": col["BESS_MW"],                                # [MW] moc ładowania/rozładowania
    "e_nom": col["BESS_MWh"],                               # [MWh] pojemność
    "efficiency_store": col["BESS_efektywnosc_magaz"],      # sprawność ładowania
    "efficiency_dispatch": col["BESS_efektywnosc_rozlad"],  # sprawność rozładowania
    "standing_loss": 0.0,                                   # straty postojowe
    "soc_initial": col["BES_initial_SoC"],                  # początkowy SOC
    "cyclic": False,                                        # wymagany SOC końcowy = startowy?
    "p_min_pu": -1.0,                                       # minimalna moc względna (ładowanie)
    "p_max_pu": 1.0,                                        # maks. moc względna (rozładowanie)
    "marginal_cost": 0.0,                                   #koszt krańcowy (degradacji TODO: sprawdź)
    "capital_cost": 0.0,                                    #CAPEX
    "carrier": "battery",                                   #typ carriera TODO: sprawdź
}

# PV i wiatr
pv_p_nom = col["PV_MW"]
wind_p_nom = col["WIND_MW"]

# flaga trybu (np. 0 = tylko autokonsumpcja, 1 = eksport do sieci)
allow_export = True #bool(col["scenario"])

bess_config, pv_p_nom, wind_p_nom, allow_export


({'name': 'battery',
  'p_nom': np.float64(10.0),
  'e_nom': np.float64(20.0),
  'efficiency_store': np.float64(1.0),
  'efficiency_dispatch': np.float64(1.0),
  'standing_loss': 0.0,
  'soc_initial': np.float64(0.0),
  'cyclic': False,
  'p_min_pu': -1.0,
  'p_max_pu': 1.0,
  'marginal_cost': 0.0,
  'capital_cost': 0.0,
  'carrier': 'battery'},
 np.float64(5.0),
 np.float64(3.0),
 True)

In [4]:
df_inp

,Data,Zapotrzebowanie na energię elektryczną [MWh/h],Fixing I cena [PLN/MWh],S1. Generacja pv [MW],S2. Generacja pv [MW],S3. Generacja pv [MW],S4. Generacja pv [MW],S5. Generacja pv [MW],S1. Generacja wind [MW],S2. Generacja wind [MW],S3. Generacja wind [MW],S4. Generacja wind [MW],S5. Generacja wind [MW]
0,2025-01-01 00:00:00,2.139788,236.617536,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.279989,4.266629,4.266629
1,2025-01-01 01:00:00,2.303984,236.607515,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.137236,3.790788,3.790788
2,2025-01-01 02:00:00,2.234366,213.087064,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.027298,3.424326,3.424326
3,2025-01-01 03:00:00,2.345014,212.455710,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.030244,3.434147,3.434147
4,2025-01-01 04:00:00,2.563971,212.455710,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.073824,3.579413,3.579413
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00,2.994140,380.816839,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.710989,5.703296,5.703296
8756,2025-12-31 20:00:00,2.909692,292.617655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.680469,5.601564,5.601564
8757,2025-12-31 21:00:00,2.642624,250.537394,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.624340,5.414466,5.414466
8758,2025-12-31 22:00:00,2.948090,80.161945,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.575964,5.253213,5.253213


In [5]:
# ---- Dane czasowe i ceny ----
hours = df_inp['Data']

prices = pd.Series(df_inp["Fixing I cena [PLN/MWh]"], index=hours)
demand_profile_mw = pd.Series(df_inp["Zapotrzebowanie na energię elektryczną [MWh/h]"], index=hours)

wind_profile_mw = pd.Series(df_inp[f"S{scenario_id}. Generacja pv [MW]"], index=hours)
pv_profile_mw = pd.Series(df_inp[f"S{scenario_id}. Generacja wind [MW]"], index=hours)


bess_config = {
    "name": "battery",
    "p_nom": 10.0,                  # [MW] moc ładowania/rozładowania
    "e_nom": 20.0,                  # [MWh] pojemność
    "efficiency_store": 1.0,        # sprawność ładowania
    "efficiency_dispatch": 1.0,     # sprawność rozładowania
    "standing_loss": 0.0,           # straty postojowe
    "soc_initial": 0.0,             # początkowy SOC
    "cyclic": False,                # wymagany SOC końcowy = startowy?
    "p_min_pu": -1.0,               # minimalna moc względna (ładowanie)
    "p_max_pu": 1.0,                # maks. moc względna (rozładowanie)
    "marginal_cost": 0.0,           # koszt degradacji
    "capital_cost": 0.0,            # CAPEX
    "carrier": "battery"
}

In [6]:
def build_bess(network, bess_config):

    if bess_config is not None:

        # wyliczenie max_hours = e_nom / p_nom
        max_hours = bess_config["e_nom"] / bess_config["p_nom"]

        network.add("StorageUnit", bess_config["name"],
                bus="node",
                p_nom=bess_config["p_nom"],
                max_hours=max_hours,

                efficiency_store=bess_config["efficiency_store"],
                efficiency_dispatch=bess_config["efficiency_dispatch"],
                standing_loss=bess_config["standing_loss"],

                state_of_charge_initial=bess_config["soc_initial"],
                cyclic_state_of_charge=bess_config["cyclic"],

                p_min_pu=bess_config["p_min_pu"],
                p_max_pu=bess_config["p_max_pu"],

                marginal_cost=bess_config["marginal_cost"],
                capital_cost=bess_config["capital_cost"],

                carrier=bess_config["carrier"]
                )
        return network

In [17]:
def build_network(bess_config, hours, demand_profile_mw, pv_profile_mw, wind_profile_mw, allow_export = False) -> pypsa.Network:
      n = pypsa.Network()
      n.set_snapshots(hours)

      # --- Bus prosumenta ---
      n.add("Bus", "node", carrier="electricity")

      # --- Load lokalny ---
      n.add("Load", "demand",
            bus="node",
            p_set=demand_profile_mw,
            carrier="electricity")

      n = build_bess(n, bess_config)

      # --- Grid jako generator z ceną energii ---
      # allow_export:
      #   False -> grid może tylko importować (p >= 0)
      #   True  -> grid może importować i eksportować (p może być ujemne)
      p_min_pu = -1.0 if allow_export else 0.0

      n.add("Generator", "grid",
            bus="node",
            p_nom=1e3,               # "nieskończona" moc
            p_min_pu=p_min_pu,
            p_max_pu=1.0,
            marginal_cost=0.0,       # nadpiszemy serię
            carrier="grid")
      n.generators_t.marginal_cost.loc[:, "grid"] = prices

      n.add("Generator", "pv",
            bus="node",
            p_nom=1e6,               # duży limit, żeby nie ograniczał
            p_min_pu=0.0,
            p_max_pu=1.0,
            marginal_cost=0.0,
            carrier="solar")

      # Set time-varying maximum output in MW
      n.generators_t.p_max.loc[:, "pv"] = pv_profile_mw

      n.add("Generator", "wind",
            bus="node",
            p_nom=1e6,               # duży limit, żeby nie ograniczał
            p_min_pu=0.0,
            p_max_pu=1.0,
            marginal_cost=0.0,
            carrier="solar")

      # Set time-varying maximum output in MW
      n.generators_t.p_max_pu.loc[:, "wind"] = wind_profile_mw

      return n

In [18]:
# --------- SCENARIUSZ 2: autokonsumpcja + eksport ---------
n_market = build_network(bess_config, hours, demand_profile_mw, pv_profile_mw, wind_profile_mw, allow_export=True)
n_market.optimize(solver_name="highs")

bess_dis_m = n_market.storage_units_t.p_dispatch["battery"]
bess_ch_m = n_market.storage_units_t.p_store["battery"]
grid_p_m = n_market.generators_t.p["grid"]

df_market = pd.DataFrame({
    "Load (MW)": n_market.loads_t.p["demand"],
    "Ładowanie BESS (MW)": bess_ch_m,
    "Rozładowanie BESS (MW)": bess_dis_m,
    "Netto BESS -> bus (MW)": bess_dis_m - bess_ch_m,
    "Import z gridu (MW)": grid_p_m,  # >0 import, <0 eksport
    "SOC BESS (MWh)": n_market.storage_units_t.state_of_charge["battery"],
})

# eksport = ujemna część wymiany z gridem (zamieniona na dodatnią)
df_market["Eksport z BESS na rynek (MW)"] = (-grid_p_m).clip(lower=0.0)
df_market["Ceny [PLN/MWh]"] = prices

mask = abs(df_market["Ładowanie BESS (MW)"] - df_market["Rozładowanie BESS (MW)"]) < 1e-6
df_market.loc[mask, ["Ładowanie BESS (MW)", "Rozładowanie BESS (MW)"]] = 0
df_market

ValueError: Series p_set has an index which does not align with the passed network snapshots.

In [12]:

# --------- SCENARIUSZ 1: tylko autokonsumpcja (bez eksportu) ---------
n_auto = build_network(bess_config, hours, demand_profile_mw, allow_export=False)
n_auto.optimize(solver_name="highs")

# dodatnie p_dispatch = rozładowanie, dodatnie p_store = ładowanie
bess_dis_auto = n_auto.storage_units_t.p_dispatch["battery"]
bess_ch_auto = n_auto.storage_units_t.p_store["battery"]
grid_p_auto = n_auto.generators_t.p["grid"]

df_auto = pd.DataFrame({
    "Load (MW)": n_auto.loads_t.p["demand"],
    "Ładowanie BESS (MW)": bess_ch_auto,
    "Rozładowanie BESS (MW)": bess_dis_auto,
    "Netto BESS -> bus (MW)": bess_dis_auto - bess_ch_auto,
    "Import z gridu (MW)": grid_p_auto,      # >= 0 w tym scenariuszu
    "SOC BESS (MWh)": n_auto.storage_units_t.state_of_charge["battery"],
    "Eksport z BESS na rynek (MW)": 0.0      # brak eksportu w tym wariancie
})

TypeError: build_network() missing 2 required positional arguments: 'pv_profile_mw' and 'wind_profile_mw'

In [14]:
# Tworzymy sieć
n = pypsa.Network()

# Dodajemy krok czasowy (np. godzina)
hours = pd.date_range("2024-01-01", "2024-01-02", freq="h", inclusive="left")
n.set_snapshots(hours)

# Dodajemy węzeł sieci z nośnikiem
n.add("Bus", "node", carrier="electricity")

# Dodajemy odbiorcę końcowego (load)
n.add("Load", "demand",
      bus="node",
      p_set=5,  # 5 MW stałego zapotrzebowania
      carrier="electricity")


# n.add("Load", "market",
#       bus="node",
#       p_set=1e6,  # 5 MW stałego zapotrzebowania
#       carrier="electricity")


# Ceny energii np. z rynku spot
prices = pd.Series([150, 100, 60, 50, 70, 120, 200, 300, 150, 80,
                    60, 50, 55, 70, 90, 110, 180, 220, 150, 100,
                    70, 60, 80, 120], index=hours)


# Dodajemy generator "market" z nośnikiem i tymczasowym kosztem
n.add("Generator", "market",
      bus="node",
      p_nom=1e6,            # bardzo wysoki limit mocy, "praktycznie" nieograniczony
      marginal_cost=0)


# Przypisujemy ceny jako koszty zmienne generatora
n.generators_t.marginal_cost.loc[:, "market"] = prices

# Dodajemy magazyn energii (BESS)
n.add("Store", "battery",
      bus="node",
      e_nom=10,     # pojemność 10 MWh
      p_nom=5,      # maksymalna moc 5 MW
      e_cyclic=True,
      # standing_loss=0.01,
      # efficiency_store=1,
      # efficiency_dispatch=1,
      carrier="battery")

# Dodajemy koszt inwestycyjny magazynu (opcjonalnie)
# n.stores.loc["battery", "capital_cost"] = 1000

# Optymalizacja systemu
n.optimize(solver_name="highs")

pd.set_option('display.float_format', '{:.2f}'.format)

# Wyświetlamy wyniki
print("Import z rynku (MW):")
print(n.generators_t.p["market"])

print("\nPoziom energii w magazynie (MWh):")
print(n.stores_t.e["battery"])

Index(['node'], dtype='object', name='name')
Index(['demand'], dtype='object', name='name')
Index(['battery'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 72 primals, 144 duals
Objective: 9.22e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


Import z rynku (MW):
snapshot
2024-01-01 00:00:00   -0.00
2024-01-01 01:00:00    5.00
2024-01-01 02:00:00    5.00
2024-01-01 03:00:00   15.00
2024-01-01 04:00:00    5.00
2024-01-01 05:00:00    5.00
2024-01-01 06:00:00   -0.00
2024-01-01 07:00:00   -0.00
2024-01-01 08:00:00    5.00
2024-01-01 09:00:00    5.00
2024-01-01 10:00:00    5.00
2024-01-01 11:00:00   15.00
2024-01-01 12:00:00    5.00
2024-01-01 13:00:00    5.00
2024-01-01 14:00:00    5.00
2024-01-01 15:00:00    5.00
2024-01-01 16:00:00   -0.00
2024-01-01 17:00:00   -0.00
2024-01-01 18:00:00    5.00
2024-01-01 19:00:00    5.00
2024-01-01 20:00:00    5.00
2024-01-01 21:00:00   15.00
2024-01-01 22:00:00    5.00
2024-01-01 23:00:00   -0.00
Freq: h, Name: market, dtype: float64

Poziom energii w magazynie (MWh):
snapshot
2024-01-01 00:00:00   -0.00
2024-01-01 01:00:00   -0.00
2024-01-01 02:00:00   -0.00
2024-01-01 03:00:00   10.00
2024-01-01 04:00:00   10.00
2024-01-01 05:00:00   10.00
2024-01-01 06:00:00    5.00
2024-01-01 07:00:00 

In [15]:
import pypsa
import pandas as pd

def build_network(allow_market_export=False):
    n = pypsa.Network()

    hours = pd.date_range("2024-01-01", "2024-01-02",
                          freq="h", inclusive="left")
    n.set_snapshots(hours)

    # --- Busy ---
    n.add("Bus", "node", carrier="electricity")
    n.add("Bus", "batt_bus", carrier="electricity")

    if allow_market_export:
        n.add("Bus", "market_bus", carrier="electricity")

    # --- Local load (autokonsumpcja dotyczy właśnie tego) ---
    n.add("Load", "demand",
          bus="node",
          p_set=5,            # MW
          carrier="electricity")

    # --- Ceny energii z gridu (import) ---
    prices = pd.Series(
        [150, 200, 250, 190, 70, 120, 200, 300, 150, 80,
         60, 50, 55, 70, 250, 350, 400, 220, 150, 100,
         70, 60, 80, 120],
        index=hours
    )

    # --- Grid: import do node ---
    n.add("Generator", "grid_import",
          bus="node",
          p_nom=1e6,
          marginal_cost=0)   # tymczasowo

    n.generators_t.marginal_cost.loc[:, "grid_import"] = prices

    # --- BESS na osobnym busie ---
    n.add("Store", "battery",
          bus="batt_bus",
          e_nom=20,       # MWh
          p_nom=10,        # MW
          e_cyclic=True,
          standing_loss=0.0,
          efficiency_store=1,
          efficiency_dispatch=1,
          carrier="battery")

    # Link node <-> bateria (ładowanie i rozładowanie na lokalny system)
    n.add("Link", "node_to_batt",
          bus0="node", bus1="batt_bus",
          p_nom=10,
          efficiency=1.0)

    n.add("Link", "batt_to_node",
          bus0="batt_bus", bus1="node",
          p_nom=10,
          efficiency=1.0)

    # --- DODATKOWO: eksport na rynek tylko z BESS ---
    if allow_market_export:
        # Load "market" – tu schodzi energia, ale tylko z batt_bus
        n.add("Load", "market_load",
              bus="market_bus",
              p_set=1e6,
              carrier="electricity")

        # Link z baterii na rynek
        n.add("Link", "batt_to_market",
              bus0="batt_bus", bus1="market_bus",
              p_nom=5,
              efficiency=1.0)

        # Generator 'export' z UJEMNYM kosztem = przychód ze sprzedaży
        # (tutaj przyjmuję te same ceny co za import, ale możesz dać inne)
        n.add("Generator", "market_export",
              bus="market_bus",
              p_nom=1e6,
              marginal_cost=0)

        n.generators_t.marginal_cost.loc[:, "market_export"] = -prices
        # minus -> przychód, bo PyPSA minimalizuje koszty

    return n


In [16]:
market_export = True

n1 = build_network(allow_market_export=market_export)
n1.optimize(solver_name="highs")

df = pd.DataFrame({
    "Load (MW)": n1.loads_t.p["demand"],
    "Ładowanie/Rozładowanie BESS [MW]": n1.stores_t.p["battery"],
    "Import z gridu (MW)": n1.generators_t.p["grid_import"],
    "Stan BESS (MWh)": n1.stores_t.e["battery"],
})

if "batt_to_market" in n1.links.index:
    df["Eksport z BESS na rynek (MW)"] = n1.links_t.p0["batt_to_market"]

df

Index(['node', 'batt_bus', 'market_bus'], dtype='object', name='name')
Index(['demand', 'market_load'], dtype='object', name='name')
Index(['node_to_batt', 'batt_to_node', 'batt_to_market'], dtype='object', name='name')
Index(['battery'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 168 primals, 384 duals
Objective: -3.74e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


,Load (MW),Ładowanie/Rozładowanie BESS [MW],Import z gridu (MW),Stan BESS (MWh),Eksport z BESS na rynek (MW)
snapshot,,,,,
2024-01-01 00:00:00,5.00,5.00,-0.00,15.00,-0.00
2024-01-01 01:00:00,5.00,5.00,-0.00,10.00,-0.00
2024-01-01 02:00:00,5.00,5.00,-0.00,5.00,-0.00
2024-01-01 03:00:00,5.00,5.00,-0.00,-0.00,-0.00
2024-01-01 04:00:00,5.00,-10.00,15.00,10.00,-0.00
2024-01-01 05:00:00,5.00,-5.00,10.00,15.00,-0.00
2024-01-01 06:00:00,5.00,5.00,-0.00,10.00,-0.00
2024-01-01 07:00:00,5.00,5.00,-0.00,5.00,-0.00
2024-01-01 08:00:00,5.00,5.00,-0.00,-0.00,-0.00


In [4]:

snapshots = n.snapshots
store_p = n.stores_t.p["battery"]       # moc ładowania (+) / rozładowania (-)
store_e = n.stores_t.e["battery"]       # stan energii
prices = n.generators_t.marginal_cost["market"]  # ceny energii
print("Import z rynku (MW):")
market_p = n.generators_t.p["market"]

# Złóż wszystko w jedną tabelę
df = pd.DataFrame({
    "snapshot": snapshots,
    "store_p (MW)": store_p.values,
    "store_e (MWh)": store_e.values,
    "market_p (MWh)": market_p.values,
    "price (EUR/MWh)": prices.values,
})

# Wyświetl tabelę
df


Import z rynku (MW):


,snapshot,store_p (MW),store_e (MWh),market_p (MWh),price (EUR/MWh)
0,2024-01-01 00:00:00,5.00,-0.00,-0.00,150
1,2024-01-01 01:00:00,-0.00,-0.00,5.00,100
2,2024-01-01 02:00:00,-0.00,-0.00,5.00,60
3,2024-01-01 03:00:00,-10.00,10.00,15.00,50
4,2024-01-01 04:00:00,-0.10,10.00,5.10,70
5,2024-01-01 05:00:00,-0.10,10.00,5.10,120
6,2024-01-01 06:00:00,4.85,5.05,0.15,200
7,2024-01-01 07:00:00,5.00,-0.00,-0.00,300
8,2024-01-01 08:00:00,-0.00,-0.00,5.00,150
9,2024-01-01 09:00:00,-0.00,-0.00,5.00,80


In [57]:
import pypsa
import pandas as pd

# Tworzymy sieć
n = pypsa.Network()

# Kroki czasowe
hours = pd.date_range("2024-01-01", "2024-01-02", freq="h", inclusive="left")
n.set_snapshots(hours)

# Węzeł sieci
n.add("Bus", "node", carrier="electricity")

# Ceny energii
prices = pd.Series([150, 100, 60, 50, 70, 120, 200, 300, 150, 80,
                    60, 50, 55, 70, 90, 110, 180, 220, 150, 100,
                    70, 60, 80, 120], index=hours)

# Generator do kupowania energii (koszt = cena)
n.add("Generator", "market_buy",
      bus="node",
      p_nom=1e6,
      marginal_cost=0)
n.generators_t.marginal_cost.loc[:, "market_buy"] = prices

# Generator do sprzedawania energii (ujemny koszt = zysk)
n.add("Generator", "market_sell",
      bus="node",
      p_nom=1e6,
      marginal_cost=0)
n.generators_t.marginal_cost.loc[:, "market_sell"] = -prices

n.add("Load", "demand",
      bus="node",
      p_set=5,  # 5 MW stałego zapotrzebowania
      carrier="electricity")

# Magazyn energii
n.add("Store", "battery",
      bus="node",
      e_nom=10,     # pojemność 10 MWh
      p_nom=5,      # maksymalna moc 5 MW
      e_cyclic=True,
      standing_loss=0.00,
      efficiency_store=1,
      efficiency_dispatch=1,
      carrier="battery")

# Optymalizacja
n.optimize(solver_name="highs")

# Wyniki
print("Moc magazynu (MW):")
print(n.stores_t.p["battery"])

print("\nStan energii (MWh):")
print(n.stores_t.e["battery"])

print("\nKupno z rynku (MW):")
print(n.generators_t.p["market_buy"])

print("\nSprzedaż na rynek (MW):")
print(n.generators_t.p["market_sell"])

Index(['battery'], dtype='object', name='Store')
Index(['demand'], dtype='object', name='Load')
Index(['node'], dtype='object', name='Bus')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.09s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 96 primals, 192 duals
Objective: -1.84e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


Moc magazynu (MW):
snapshot
2024-01-01 00:00:00   -10.00
2024-01-01 01:00:00    -0.00
2024-01-01 02:00:00     5.00
2024-01-01 03:00:00     5.00
2024-01-01 04:00:00    -0.00
2024-01-01 05:00:00    -0.00
2024-01-01 06:00:00    -0.00
2024-01-01 07:00:00   -10.00
2024-01-01 08:00:00    -0.00
2024-01-01 09:00:00    -0.00
2024-01-01 10:00:00    -0.00
2024-01-01 11:00:00     5.00
2024-01-01 12:00:00     5.00
2024-01-01 13:00:00    -0.00
2024-01-01 14:00:00    -0.00
2024-01-01 15:00:00    -0.00
2024-01-01 16:00:00    -0.00
2024-01-01 17:00:00   -10.00
2024-01-01 18:00:00    -0.00
2024-01-01 19:00:00    -0.00
2024-01-01 20:00:00     5.00
2024-01-01 21:00:00     5.00
2024-01-01 22:00:00    -0.00
2024-01-01 23:00:00    -0.00
Freq: h, Name: battery, dtype: float64

Stan energii (MWh):
snapshot
2024-01-01 00:00:00   10.00
2024-01-01 01:00:00   10.00
2024-01-01 02:00:00    5.00
2024-01-01 03:00:00   -0.00
2024-01-01 04:00:00   -0.00
2024-01-01 05:00:00   -0.00
2024-01-01 06:00:00   -0.00
2024-01-01 